# GenomicSuperSignature - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [1]:
library(here)
library(matrixStats)
library(factoextra)
library(cluster)
library(GenomicSuperSignature)

source(here("config.R"))
set.seed(config$GTEx$RANDOM_SVD_SEED)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: ggplot2

Welcome! Want to learn more? See two factoextra-related books at https://goo.gl/ve3WBa

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQR

In [2]:
input_dir <- config$GTEx$OUTPUT_DIR
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3

In [3]:
gtex_data <- readRDS(here('output/gtex/df_gtex_fbm_filt.rds'))
n <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))
study <- 'GTEx'
d <- 4

message("Data dimensions: ", nrow(gtex_data), " genes x ", ncol(gtex_data), " samples")
message("Number of PCs: ", n)

Data dimensions: 21613 genes x 17382 samples

Number of PCs: 412



In [4]:
GenomicSuperSignature_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("GenomicSuperSignature run", i, "of", N_RUNS, "\n")
  
  start_time <- Sys.time()
  
  # PCA
  pca_res <- prcomp(t(as.matrix(gtex_data)))
  
  trainingData_PCA <- list()
  trainingData_PCA[[study]] <- list()
  trainingData_PCA[[study]]$rotation <- pca_res$rotation[, 1:n]
  colnames(trainingData_PCA[[study]]$rotation) <- paste0(study, ".PC", 1:n)
  
  eigs <- pca_res$sdev^2
  pca_summary <- rbind(SD = sqrt(eigs),
                       Variance = eigs/sum(eigs),
                       Cumulative = cumsum(eigs)/sum(eigs))
  trainingData_PCA[[study]]$variance <- pca_summary[,1:n]
  colnames(trainingData_PCA[[study]]$variance) <- paste0(study, ".PC", c(1:n))
  
  # Hierarchical clustering
  allZ <- trainingData_PCA[[study]]$rotation
  storage.mode(allZ) <- "double"
  all  <- t(allZ)
  
  res.dist <- factoextra::get_dist(all, method = "spearman")
  
  k <- round(nrow(all)/d, 0)
  res.hcut <- factoextra::hcut(res.dist, k = k, hc_func = "hclust", 
                               hc_method = "ward.D", hc_metric = "spearman")
  
  # Build avgLoading 
  trainingData_PCclusters <- buildAvgLoading(allZ, k, cluster = res.hcut$cluster)
  
  # Silhouette Width
  cl <- trainingData_PCclusters$cluster
  silh_res <- cluster::silhouette(cl, res.dist)
  cl_silh_width <- summary(silh_res)$clus.avg.widths
  trainingData_PCclusters$sw <- cl_silh_width
  
  # Final model
  trainingData_df <- DataFrame(
    PCAsummary = I(list(trainingData_PCA[[study]]$variance))
  )
  rownames(trainingData_df) <- study
  
  RAVmodel <- PCAGenomicSignatures(
    assays       = list(RAVindex = as.matrix(trainingData_PCclusters$avgLoading)),
    trainingData = trainingData_df
  )
  
  metadata(RAVmodel) <- trainingData_PCclusters[c("cluster","size","k","n")]
  names(metadata(RAVmodel)$size) <- paste0("RAV", seq_len(ncol(RAVmodel)))
  geneSets(RAVmodel)        <- "Custom"
  studies(RAVmodel)         <- trainingData_PCclusters$studies
  silhouetteWidth(RAVmodel) <- trainingData_PCclusters$sw
  updateNote(RAVmodel)      <- paste0("Single-matrix GTEx model; PCs = ", n, ".")
  metadata(RAVmodel)$version <- "0.1.0-single"
  
  end_time <- Sys.time()
  GenomicSuperSignature_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", GenomicSuperSignature_times[i], "minutes\n\n")
}

GenomicSuperSignature run 1 of 3 
Run 1 time: 27.78381 minutes

GenomicSuperSignature run 2 of 3 
Run 2 time: 26.05914 minutes

GenomicSuperSignature run 3 of 3 
Run 3 time: 28.01627 minutes



In [5]:
GenomicSuperSignature_time_minutes <- GenomicSuperSignature_times
names(GenomicSuperSignature_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(GenomicSuperSignature_time_minutes, file.path(output_dir, "GenomicSuperSignature_time_minutes.rds"))
cat("GenomicSuperSignature times:", GenomicSuperSignature_time_minutes, "minutes\n")

GenomicSuperSignature times: 27.78381 26.05914 28.01627 minutes
